# Module 7 — Handling Imbalanced Data: SMOTE, ADASYN & Weighted Loss

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, Imbalanced-learn, PyTorch, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 6 (Statistical Tests)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Imbalanced Dataset](#3-synthetic-imbalanced-dataset)
4. [The Problem: Why Accuracy is Misleading](#4-the-problem-why-accuracy-is-misleading)
5. [Oversampling: SMOTE & ADASYN](#5-oversampling-smote--adasyn)
6. [Undersampling: Tomek Links](#6-undersampling-tomek-links)
7. [Class Weights in Scikit-learn](#7-class-weights-in-scikit-learn)
8. [Weighted Cross-Entropy in PyTorch](#8-weighted-cross-entropy-in-pytorch)
9. [Comparison Dashboard](#9-comparison-dashboard)
10. [Key Business Takeaway](#10-key-business-takeaway)

---
## §1 · Business Context

### The Imbalanced Data Problem at Revolut

Fraud detection is the **classic imbalanced classification task**:

| Statistic | Value |
|-----------|-------|
| Total transactions per day | 10,000,000 |
| Fraudulent transactions | ~10,000 (0.1%) |
| **Class ratio** | **1:1000** |

**Why this matters**:
- A model that predicts "not fraud" for every transaction achieves **99.9% accuracy** — but catches **zero fraud**.
- Standard ML algorithms optimize for accuracy and ignore the minority class.
- **Business cost**: missing a £5,000 fraudulent transaction is 100× worse than flagging a legitimate one for review.

**Solutions we will explore**:
1. **Oversampling** the minority class (SMOTE, ADASYN)
2. **Undersampling** the majority class (Tomek Links)
3. **Class weights** in Scikit-learn models
4. **Weighted loss functions** in PyTorch

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)

# ── Imbalanced-learn ─────────────────────────────────────────────
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

# ── PyTorch ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Imbalanced Dataset

We simulate a **100,000-transaction fraud dataset** with **1% positive rate** (realistic for fintech).

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TOTAL = 100_000
FRAUD_RATE = 0.01  # 1% fraud rate
N_FRAUD = int(N_TOTAL * FRAUD_RATE)
N_LEGIT = N_TOTAL - N_FRAUD

print(f"Generating imbalanced dataset:")
print(f"  Total transactions: {N_TOTAL:,}")
print(f"  Fraud (positive)  : {N_FRAUD:,} ({FRAUD_RATE*100:.1f}%)")
print(f"  Legit (negative)  : {N_LEGIT:,} ({(1-FRAUD_RATE)*100:.1f}%)")
print(f"  Class ratio       : 1:{N_LEGIT // N_FRAUD}")

# ── Feature generation ───────────────────────────────────────────
# 20 features: some discriminative, some noise
n_features = 20

# Legitimate transactions
X_legit = np.random.randn(N_LEGIT, n_features)
X_legit[:, 0] = np.random.lognormal(3, 1, N_LEGIT)  # amount
X_legit[:, 1] = np.random.uniform(0, 24, N_LEGIT)   # hour
X_legit[:, 2] = np.random.normal(0, 1, N_LEGIT)     # velocity

# Fraudulent transactions (shifted distributions)
X_fraud = np.random.randn(N_FRAUD, n_features)
X_fraud[:, 0] = np.random.lognormal(5, 1.5, N_FRAUD)  # higher amounts
X_fraud[:, 1] = np.random.uniform(20, 24, N_FRAUD)    # late-night bias
X_fraud[:, 2] = np.random.normal(3, 1.5, N_FRAUD)     # higher velocity
X_fraud[:, 3] = np.random.normal(2, 1, N_FRAUD)       # geo-anomaly

X = np.vstack([X_legit, X_fraud])
y = np.concatenate([np.zeros(N_LEGIT), np.ones(N_FRAUD)])

# Shuffle
idx = np.random.permutation(len(y))
X = X[idx]
y = y[idx]

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(n_features)])
df["is_fraud"] = y

print(f"\n✓ Shape: {df.shape}")
print(f"  Fraud rate: {y.mean() * 100:.2f}%")
df.head()

In [ ]:
# Visualize class imbalance
fig = go.Figure()
fig.add_trace(go.Pie(
    labels=["Legitimate", "Fraud"],
    values=[N_LEGIT, N_FRAUD],
    marker=dict(colors=["#00CC96", "#EF553B"]),
    hole=0.4,
))
fig.update_layout(title="Class Distribution (1% Fraud Rate)", height=380)
fig.show()

---
## §4 · The Problem: Why Accuracy is Misleading

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Train: {len(y_train):,} samples ({y_train.sum():.0f} fraud, {y_train.sum()/len(y_train)*100:.2f}%)")
print(f"Test : {len(y_test):,} samples ({y_test.sum():.0f} fraud, {y_test.sum()/len(y_test)*100:.2f}%)")

# ── Baseline: "Always predict not fraud" ─────────────────────────
y_pred_all_negative = np.zeros(len(y_test))

print("\n" + "=" * 60)
print("BASELINE: Predict All Negative (No Fraud)")
print("=" * 60)
print(f"Accuracy : {accuracy_score(y_test, y_pred_all_negative):.4f}  ← looks great!")
print(f"Precision: {precision_score(y_test, y_pred_all_negative, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_all_negative, zero_division=0):.4f}  ← catches 0 fraud!")
print(f"F1-Score : {f1_score(y_test, y_pred_all_negative, zero_division=0):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_pred_all_negative):.4f}")

print("\n⚠️  Accuracy = 99% but Recall = 0% → the model is useless for fraud detection!")
print("   This is why we need better metrics: PR-AUC, F1, or business-specific cost metrics.")

---
## §5 · Oversampling: SMOTE & ADASYN

### 5.1 — SMOTE (Synthetic Minority Oversampling Technique)

SMOTE generates synthetic fraud examples by interpolating between existing fraud cases.

In [ ]:
# Apply SMOTE to training data only (never to test!)
smote = SMOTE(sampling_strategy=0.3, random_state=42)  # oversample to 30% of majority
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("=" * 60)
print("SMOTE Oversampling Results")
print("=" * 60)
print(f"Before SMOTE:")
print(f"  Train size : {len(y_train):,}")
print(f"  Fraud count: {y_train.sum():.0f} ({y_train.mean()*100:.2f}%)")
print(f"\nAfter SMOTE:")
print(f"  Train size : {len(y_train_smote):,}")
print(f"  Fraud count: {y_train_smote.sum():.0f} ({y_train_smote.mean()*100:.2f}%)")
print(f"  New fraud ratio: 1:{int((1-y_train_smote.mean())/y_train_smote.mean())}")

# Visualize
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Before SMOTE", "After SMOTE"])

fig.add_trace(go.Pie(labels=["Legit", "Fraud"],
                     values=[(y_train==0).sum(), (y_train==1).sum()],
                     marker=dict(colors=["#00CC96", "#EF553B"]), hole=0.4), row=1, col=1)

fig.add_trace(go.Pie(labels=["Legit", "Fraud"],
                     values=[(y_train_smote==0).sum(), (y_train_smote==1).sum()],
                     marker=dict(colors=["#00CC96", "#EF553B"]), hole=0.4), row=1, col=2)

fig.update_layout(height=380, showlegend=False)
fig.show()

### 5.2 — ADASYN (Adaptive Synthetic Sampling)

ADASYN focuses on **hard-to-learn** minority examples near the decision boundary.

In [ ]:
# Apply ADASYN
adasyn = ADASYN(sampling_strategy=0.3, random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)

print("=" * 60)
print("ADASYN Oversampling Results")
print("=" * 60)
print(f"After ADASYN:")
print(f"  Train size : {len(y_train_adasyn):,}")
print(f"  Fraud count: {y_train_adasyn.sum():.0f} ({y_train_adasyn.mean()*100:.2f}%)")

print("\n💡 ADASYN vs. SMOTE:")
print("   - SMOTE: uniform synthetic samples across minority class")
print("   - ADASYN: more samples near decision boundary (harder cases)")
print("   - ADASYN can be more effective but also more sensitive to noise")

---
## §6 · Undersampling: Tomek Links

Tomek Links remove majority-class samples that are "close" to minority samples,
cleaning the decision boundary.

In [ ]:
# Apply Tomek Links
tomek = TomekLinks()
X_train_tomek, y_train_tomek = tomek.fit_resample(X_train, y_train)

print("=" * 60)
print("Tomek Links Undersampling Results")
print("=" * 60)
print(f"Before Tomek:")
print(f"  Train size : {len(y_train):,}")
print(f"  Legit count: {(y_train==0).sum():,}")
print(f"\nAfter Tomek:")
print(f"  Train size : {len(y_train_tomek):,}")
print(f"  Legit count: {(y_train_tomek==0).sum():,}")
print(f"  Removed    : {len(y_train) - len(y_train_tomek):,} borderline legit samples")

print("\n💡 Tomek Links clean the boundary but don't increase fraud samples")
print("   Best combined with SMOTE (SMOTETomek)")

# Combined: SMOTE + Tomek
smote_tomek = SMOTETomek(random_state=42)
X_train_combined, y_train_combined = smote_tomek.fit_resample(X_train, y_train)

print(f"\nSMOTE + Tomek combined:")
print(f"  Train size : {len(y_train_combined):,}")
print(f"  Fraud ratio: {y_train_combined.mean()*100:.2f}%")

---
## §7 · Class Weights in Scikit-learn

Instead of resampling data, we can tell the model to **penalize mistakes on the minority class more heavily**.

In [ ]:
# Train logistic regression with class_weight='balanced'
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Without class weights
lr_no_weight = LogisticRegression(max_iter=1000, random_state=42)
lr_no_weight.fit(X_train_scaled, y_train)
y_pred_no_weight = lr_no_weight.predict(X_test_scaled)
y_prob_no_weight = lr_no_weight.predict_proba(X_test_scaled)[:, 1]

# With balanced class weights
lr_balanced = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_balanced.fit(X_train_scaled, y_train)
y_pred_balanced = lr_balanced.predict(X_test_scaled)
y_prob_balanced = lr_balanced.predict_proba(X_test_scaled)[:, 1]

print("=" * 70)
print("Logistic Regression: No Weights vs. Balanced Weights")
print("=" * 70)
print(f"\n{'Metric':<20} {'No Weights':<15} {'Balanced':<15}")
print("-" * 50)
print(f"{'Accuracy':<20} {accuracy_score(y_test, y_pred_no_weight):<15.4f} {accuracy_score(y_test, y_pred_balanced):<15.4f}")
print(f"{'Precision':<20} {precision_score(y_test, y_pred_no_weight):<15.4f} {precision_score(y_test, y_pred_balanced):<15.4f}")
print(f"{'Recall':<20} {recall_score(y_test, y_pred_no_weight):<15.4f} {recall_score(y_test, y_pred_balanced):<15.4f}")
print(f"{'F1-Score':<20} {f1_score(y_test, y_pred_no_weight):<15.4f} {f1_score(y_test, y_pred_balanced):<15.4f}")
print(f"{'PR-AUC':<20} {average_precision_score(y_test, y_prob_no_weight):<15.4f} {average_precision_score(y_test, y_prob_balanced):<15.4f}")
print(f"{'ROC-AUC':<20} {roc_auc_score(y_test, y_prob_no_weight):<15.4f} {roc_auc_score(y_test, y_prob_balanced):<15.4f}")

print("\n💡 Balanced weights dramatically improve recall (catch more fraud)")
print("   at the cost of lower precision (more false alarms)")

---
## §8 · Weighted Cross-Entropy in PyTorch

For neural networks, we can define a **custom loss function** that weights the minority class.

In [ ]:
# ── Build a simple neural network ────────────────────────────────
class FraudNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )
    
    def forward(self, x):
        return self.net(x).squeeze()

# ── Prepare data ─────────────────────────────────────────────────
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

# ── Compute class weights ────────────────────────────────────────
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
weight_for_0 = n_neg / (n_neg + n_pos)
weight_for_1 = n_pos / (n_neg + n_pos)
class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float32)

print(f"Class weights:")
print(f"  Negative (legit): {weight_for_0:.4f}")
print(f"  Positive (fraud): {1 / weight_for_1:.1f}× (effective)")
print(f"  → Fraud samples are weighted {n_neg / n_pos:.0f}× more heavily")

# Custom weighted BCE loss
class WeightedBCELoss(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.pos_weight = pos_weight
    
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        weights = torch.where(targets == 1, self.pos_weight, 1.0)
        return (bce * weights).mean()

pos_weight = torch.tensor(n_neg / n_pos, dtype=torch.float32)
criterion_weighted = WeightedBCELoss(pos_weight)
criterion_standard = nn.BCEWithLogitsLoss()

In [ ]:
# ── Training function ────────────────────────────────────────────
def train_model(model, criterion, n_epochs=20, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    return model

# ── Train both models ────────────────────────────────────────────
print("Training standard model …")
model_std = FraudNet(X_train_scaled.shape[1])
train_model(model_std, criterion_standard, n_epochs=20)

print("Training weighted model …")
model_weighted = FraudNet(X_train_scaled.shape[1])
train_model(model_weighted, criterion_weighted, n_epochs=20)

# ── Evaluate ─────────────────────────────────────────────────────
def evaluate_model(model, X_test, y_test, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= threshold).astype(int)
    
    return {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "PR-AUC": average_precision_score(y_test, probs),
        "ROC-AUC": roc_auc_score(y_test, probs),
    }

results_std = evaluate_model(model_std, X_test_t, y_test_t)
results_weighted = evaluate_model(model_weighted, X_test_t, y_test_t)

print("\n" + "=" * 70)
print("PyTorch Neural Network: Standard vs. Weighted Loss")
print("=" * 70)
print(f"\n{'Metric':<20} {'Standard BCE':<15} {'Weighted BCE':<15}")
print("-" * 50)
for metric in ["Accuracy", "Precision", "Recall", "F1", "PR-AUC", "ROC-AUC"]:
    print(f"{metric:<20} {results_std[metric]:<15.4f} {results_weighted[metric]:<15.4f}")

print("\n💡 Weighted loss improves recall (catches more fraud)")
print("   but may reduce precision (more false positives)")

---
## §9 · Comparison Dashboard

In [ ]:
# ── Compile all results ──────────────────────────────────────────
all_results = []

# 1. Baseline (all negative)
all_results.append({
    "Method": "Baseline (All Negative)",
    "Accuracy": accuracy_score(y_test, y_pred_all_negative),
    "Precision": precision_score(y_test, y_pred_all_negative, zero_division=0),
    "Recall": recall_score(y_test, y_pred_all_negative, zero_division=0),
    "F1": f1_score(y_test, y_pred_all_negative, zero_division=0),
    "PR-AUC": average_precision_score(y_test, y_pred_all_negative),
})

# 2. Logistic Regression (no weights)
all_results.append({
    "Method": "LR (No Weights)",
    "Accuracy": accuracy_score(y_test, y_pred_no_weight),
    "Precision": precision_score(y_test, y_pred_no_weight),
    "Recall": recall_score(y_test, y_pred_no_weight),
    "F1": f1_score(y_test, y_pred_no_weight),
    "PR-AUC": average_precision_score(y_test, y_prob_no_weight),
})

# 3. Logistic Regression (balanced)
all_results.append({
    "Method": "LR (Balanced Weights)",
    "Accuracy": accuracy_score(y_test, y_pred_balanced),
    "Precision": precision_score(y_test, y_pred_balanced),
    "Recall": recall_score(y_test, y_pred_balanced),
    "F1": f1_score(y_test, y_pred_balanced),
    "PR-AUC": average_precision_score(y_test, y_prob_balanced),
})

# 4. LR + SMOTE
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
X_train_smote_scaled = scaler.transform(X_train_smote)
lr_smote.fit(X_train_smote_scaled, y_train_smote)
y_pred_smote = lr_smote.predict(X_test_scaled)
y_prob_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]

all_results.append({
    "Method": "LR + SMOTE",
    "Accuracy": accuracy_score(y_test, y_pred_smote),
    "Precision": precision_score(y_test, y_pred_smote),
    "Recall": recall_score(y_test, y_pred_smote),
    "F1": f1_score(y_test, y_pred_smote),
    "PR-AUC": average_precision_score(y_test, y_prob_smote),
})

# 5. Neural Network (standard)
all_results.append({
    "Method": "NN (Standard BCE)",
    **results_std
})

# 6. Neural Network (weighted)
all_results.append({
    "Method": "NN (Weighted BCE)",
    **results_weighted
})

df_results = pd.DataFrame(all_results).round(4)
print("=" * 90)
print("COMPREHENSIVE COMPARISON: Imbalanced Data Techniques")
print("=" * 90)
print(df_results.to_string(index=False))

In [ ]:
# ── 9.1 PR Curves ────────────────────────────────────────────────
fig = go.Figure()

methods = [
    ("Baseline", y_pred_all_negative, None),
    ("LR (No Weights)", y_pred_no_weight, y_prob_no_weight),
    ("LR (Balanced)", y_pred_balanced, y_prob_balanced),
    ("LR + SMOTE", y_pred_smote, y_prob_smote),
]

colors = ["#636EFA", "#EF553B", "#00CC96", "#FFA15A"]

for (name, _, probs), color in zip(methods[1:], colors):
    precision, recall, _ = precision_recall_curve(y_test, probs)
    fig.add_trace(go.Scatter(
        x=recall, y=precision,
        mode="lines",
        name=f"{name}",
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title="Precision-Recall Curves (Higher is Better)",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=450,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 PR-AUC is the gold standard metric for imbalanced classification")
print("   (ROC-AUC can be misleading when the positive class is rare)")

In [ ]:
# ── 9.2 Confusion Matrix Comparison ──────────────────────────────
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["LR (No Weights)", "LR (Balanced)",
                                    "LR + SMOTE", "NN (Weighted BCE)"])

y_preds_list = [y_pred_no_weight, y_pred_balanced, y_pred_smote,
                (torch.sigmoid(model_weighted(X_test_t)) >= 0.5).int().numpy()]
titles = ["LR (No Weights)", "LR (Balanced)", "LR + SMOTE", "NN (Weighted BCE)"]

for i, (y_pred, title) in enumerate(zip(y_preds_list, titles), 1):
    cm = confusion_matrix(y_test, y_pred)
    fig.add_trace(go.Heatmap(
        z=cm, x=["Pred 0", "Pred 1"], y=["Actual 0", "Actual 1"],
        colorscale="Blues", showscale=False,
        text=cm, texttemplate="%{text}", textfont={"size": 16},
    ), row=(i-1)//2 + 1, col=(i-1) % 2 + 1)

fig.update_layout(height=600, title_text="Confusion Matrices (Higher diagonal = better)")
fig.show()

In [ ]:
# ── 9.3 Business Cost Analysis ───────────────────────────────────
# Assume:
# - Cost of false negative (missed fraud): £5,000
# - Cost of false positive (legit flagged): £50

cost_fn = 5000  # missed fraud
cost_fp = 50    # false alarm

print("=" * 70)
print("BUSINESS COST ANALYSIS")
print("=" * 70)
print(f"\nAssumptions:")
print(f"  Cost of missed fraud (FN)  : £{cost_fn:,}")
print(f"  Cost of false alarm  (FP)  : £{cost_fp:,}")
print(f"\n{'Method':<25} {'FP':>8} {'FN':>8} {'Total Cost':>12}")
print("-" * 60)

for _, row in df_results.iterrows():
    # Estimate FP and FN from confusion matrix
    method = row["Method"]
    if method == "Baseline (All Negative)":
        fp, fn = 0, int(y_test.sum())
    else:
        y_pred_method = {
            "LR (No Weights)": y_pred_no_weight,
            "LR (Balanced Weights)": y_pred_balanced,
            "LR + SMOTE": y_pred_smote,
        }.get(method, None)
        if y_pred_method is None:
            continue
        cm = confusion_matrix(y_test, y_pred_method)
        fp = cm[0, 1]
        fn = cm[1, 0]
    
    total_cost = fp * cost_fp + fn * cost_fn
    print(f"{method:<25} {fp:>8,} {fn:>8,} £{total_cost:>11,}")

print("\n💡 The best method minimizes total business cost, not just error rate")

---
## §10 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Technique | When to Use | Pros | Cons |
> |-----------|-------------|------|------|
> | **SMOTE** | Small minority class | Generates diverse synthetic samples | Can create noisy samples |
> | **ADASYN** | Hard-to-learn boundary | Focuses on difficult cases | Sensitive to outliers |
> | **Tomek Links** | Noisy majority class | Cleans decision boundary | Doesn't add minority samples |
> | **Class Weights** | Any imbalanced dataset | No data modification needed | May hurt calibration |
> | **Weighted Loss (PyTorch)** | Neural networks | Flexible, end-to-end trainable | Requires tuning |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Never use accuracy** for imbalanced fraud detection — use PR-AUC, F1, or business cost.
>
> 2. **Always stratify** train/test splits to preserve the fraud ratio:
>    ```python
>    train_test_split(X, y, stratify=y, test_size=0.3)
>    ```
>
> 3. **Resample only the training set** — never touch the test set:
>    ```python
>    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
>    # X_test remains unchanged!
>    ```
>
> 4. **Threshold tuning** is critical — the default 0.5 threshold is rarely optimal:
>    ```python
>    # Find threshold that maximizes F1
>    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
>    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
>    optimal_threshold = thresholds[np.argmax(f1_scores)]
>    ```
>
> 5. **Cost-sensitive learning** aligns the model with business objectives:
>    - Assign `cost_fn` = average fraud amount (£5,000)
>    - Assign `cost_fp` = manual review cost (£50)
>    - Optimize for `total_cost = FP * cost_fp + FN * cost_fn`
>
> ### 📌 Metric Selection Guide
>
> ```
> What matters most?
> ├─ Catching ALL fraud (even with false alarms) → Maximize RECALL
> ├─ Minimizing false alarms (even if some fraud slips through) → Maximize PRECISION
> ├─ Balance between the two → Maximize F1-SCORE
> └─ Overall ranking quality → Maximize PR-AUC
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Imbalanced data requires specialized techniques** — standard ML fails.
> 2. **PR-AUC > ROC-AUC** for rare positive classes.
> 3. **Class weights are the easiest fix** — start there before resampling.
> 4. **Always validate with business cost**, not just statistical metrics.
> 5. **Threshold tuning** is as important as the model itself.